# ICU admission Model Pipeline

Public analysis code for the **ICU admission** outcome after open laparotomy surgery.

**Data are not included.** Place an authorized, appropriately prepared dataset at `data/input_data.csv`. Required column names are documented in `data/data_dictionary.csv`.


## 1. Package imports


In [ ]:
import os
import joblib
import warnings
from collections import Counter

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from xgboost import XGBClassifier

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, auc, brier_score_loss, confusion_matrix,
    f1_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    GridSearchCV, cross_val_predict, train_test_split
)

warnings.filterwarnings("ignore")


## 2. Configuration


In [ ]:
# ==========================================
# 1. Portable file and output configuration
# ==========================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "input_data.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "icu_admission"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

Y_LABEL = 'ICU admission'
X_FEATURES = ['Age', 'Sex', 'BMI', 'Smoking', 'Emergency', 'ASA', 'eGFR', 'HCT', 'aPTT', 'Platelet', 'Protime_PT', 'ALT', 'Dependent status', 'NHI procedure points', 'DM', 'HT', 'cancer', 'kidney', 'Respiratory']
FEATURE_NAME = "primary_19_features"

# Complete GridSearchCV ranges reported in Supplementary Table 3.
# Single-value lists are fixed settings included for reproducibility.
GLOBAL_CONFIG = {
    "test_size": 0.30,
    "random_state": 55,
    "threshold": None,
    "switches": {
        "metrics": True,
        "plots": True,
        "pkl": True,
        "proba": True,
        "shap": True,
    },
    "save": {
        "metrics": True,
        "plots": True,
    },
    "params": {
        "LGBM_grid": {
            "learning_rate": [0.001, 0.01, 0.02, 0.05, 0.1],
            "n_estimators": [100, 300, 500, 700],
            "max_depth": [2, 3, 4, 5, 6],
            "num_leaves": [10, 16, 18, 31, 50],
            "min_child_samples": [15, 20, 25, 30],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
            "reg_lambda": [0, 1, 2],
            "verbosity": [-1],
            "scale_pos_weight": [3],
            "use_missing": [True],
            "random_state": [55],
        },
        "XGB_grid": {
            "learning_rate": [0.001, 0.01, 0.02, 0.05, 0.1],
            "n_estimators": [100, 300, 500, 700],
            "max_depth": [2, 3, 4, 5, 6],
            "subsample": [0.5, 0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
            "min_child_weight": [1, 2, 3, 4, 5],
            "gamma": [0],
            "reg_alpha": [0, 0.5, 1],
            "reg_lambda": [1, 2],
            "scale_pos_weight": [3],
            "random_state": [55],
        },
    },
}


## 3. Core modeling pipeline


In [ ]:
def build_lgbm_param_grid(search_space):
    """Apply the reported constraint num_leaves <= 2**max_depth."""
    shared = {
        key: values
        for key, values in search_space.items()
        if key not in {"max_depth", "num_leaves"}
    }
    valid_grids = []
    for depth in search_space["max_depth"]:
        valid_leaves = [
            leaves for leaves in search_space["num_leaves"]
            if leaves <= 2**depth
        ]
        if valid_leaves:
            valid_grids.append({
                **shared,
                "max_depth": [depth],
                "num_leaves": valid_leaves,
            })
    return valid_grids


class ModelManager:
    def __init__(self, cfg, input_path, output_root):
        self.cfg = cfg
        self.input_path = input_path
        # feature name (外部控制)
        self.feature_name = FEATURE_NAME
        # self.output_dir = os.path.join(output_root,Y_LABEL)# 根目錄
        self.output_dir =output_root
        
        # 2. 定義資料夾路徑 (修正反斜線問題，確保 os.path.join 正常運作)
        self.dirs = {
            'pkl': os.path.join(self.output_dir, 'pkl',FEATURE_NAME),
            'perf': os.path.join(self.output_dir, 'performance'),
            'plot': os.path.join(self.output_dir, 'plot',FEATURE_NAME),
            'prob': os.path.join(self.output_dir, 'Probabilities'),
            'shap': os.path.join(self.output_dir, 'shap',FEATURE_NAME),
            'data': os.path.join(self.output_dir, 'train_test_data')
        }
        
        # 自動建立所有層級的資料夾
        for d in self.dirs.values(): 
            os.makedirs(d, exist_ok=True)

    def load_data(self):
        df = pd.read_csv(self.input_path)
        X = df[X_FEATURES]
        y = df[Y_LABEL]
        print(f"Total samples: {len(y)}")
        print(f"Positive count: {(y==1).sum()}")
        print(f"Negative count: {(y==0).sum()}")
        print(f"Positive rate: {y.mean():.3f}")
        return train_test_split(X, y, test_size=self.cfg['test_size'], stratify=y, random_state=self.cfg['random_state'])

    def get_metrics(self, y_true, y_proba, label, model_name, threshold=None):
        
        if threshold is None:
            threshold = self.final_thresholds.get(model_name, 0.5)
        y_pred = (y_proba >= threshold).astype(int)
           
        # 1. 計算混淆矩陣
        cm = confusion_matrix(y_true, y_pred)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
        else:
            # 處理特殊情況 (只有單一類別預測)
            tn = cm[0, 0] if (y_true == 0).all() else 0
            tp = cm[0, 0] if (y_true == 1).all() else 0
            fp = fn = 0

        # 2. 計算基礎與臨床指標
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0 
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0 
        brier = brier_score_loss(y_true, y_proba)

        # LR+ = Sens / (1-Spec), LR- = (1-Sens) / Spec
        lr_plus = round(sensitivity / (1 - specificity), 3) if (1 - specificity) > 0 else "N/A"
        lr_minus = round((1 - sensitivity) / specificity, 3) if specificity > 0 else "N/A"
    
        return {
            'Model': model_name, 'Dataset': label, 
            'Threshold_type': 'Youden' if threshold != 0.5 else '0.5',
            'Threshold': round(threshold,4),
            'Accuracy': round(accuracy_score(y_true, y_pred), 3),
            'Sensitivity': round(sensitivity, 3),
            'Specificity': round(specificity, 3),
            'AUC': round(roc_auc_score(y_true, y_proba), 3),
            'PPV': round(ppv, 3),
            'NPV': round(npv, 3),
            'Brier_score_before': round(brier, 4),
            'LR+': lr_plus,
            'LR-': lr_minus,
            'F1': round(f1_score(y_true, y_pred, average='binary'), 3),
            'F1_micro': round(f1_score(y_true, y_pred, average='micro'), 3),
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn 
        }

    def _save_visuals(self, y_true, y_proba, model_name, label,threshold,suffix):
        y_pred = (y_proba >= threshold).astype(int)
        save_kwargs = {'dpi': 300, 'bbox_inches': 'tight'}

        # ================= Confusion Matrix =================
        plt.figure(figsize=(6, 5))
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'CM (Thres={threshold:.3f})')
        # plt.show()

        if self.cfg['save']['plots']:
            plt.savefig(
                os.path.join(self.dirs['plot'], f'{Y_LABEL}_{model_name}_{label}_CM_{suffix}_{self.feature_name}.png'),
                **save_kwargs
            )
    
        plt.close()

    def _plot_combined_curves(self, combined_data, label):
        save_kwargs = {'dpi': 300, 'bbox_inches': 'tight'}
        
        # Combined ROC
        plt.figure(figsize=(8, 7))
        for name, (y_true, y_proba) in combined_data.items():
            fpr, tpr, _ = roc_curve(y_true, y_proba)
            plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
        plt.title(f'Combined ROC Curve ({label})')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)

        if self.cfg['save']['plots']:
            plt.savefig(
                os.path.join(self.dirs['plot'],
                f'{Y_LABEL}_Combined_{label}_ROC_{self.feature_name}.png'),
                **save_kwargs
            )
        plt.show()
        plt.close()

    def _plot_shap(self, model, X_data, model_name):
        print(f"Generating SHAP plots: {model_name}")
        try:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_data)

            if isinstance(shap_values, list) and len(shap_values) == 2:
                shap_values_to_plot = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                shap_values_to_plot = shap_values[:, :, 1]
            else:
                shap_values_to_plot = shap_values

            save_kwargs = {"dpi": 300, "bbox_inches": "tight"}

            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_values_to_plot, X_data, max_display=60, show=False
            )
            plt.title(f"SHAP Summary - {model_name}")
            plt.savefig(
                os.path.join(
                    self.dirs["shap"],
                    f"{Y_LABEL}_{model_name}_SHAP_summary_{self.feature_name}.png",
                ),
                **save_kwargs,
            )
            plt.close()

            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_values_to_plot, X_data, plot_type="bar",
                max_display=60, show=False
            )
            plt.title(f"SHAP Importance - {model_name}")
            plt.savefig(
                os.path.join(
                    self.dirs["shap"],
                    f"{Y_LABEL}_{model_name}_SHAP_bar_{self.feature_name}.png",
                ),
                **save_kwargs,
            )
            plt.close()
        except Exception as exc:
            print(f"SHAP analysis failed for {model_name}: {exc}")

    def find_best_threshold(self, model, X_train, y_train):

        print(">>> Finding optimal threshold by Youden index (Train CV only)...")
    
        # out-of-fold probabilities
        oof_proba = cross_val_predict(
            clone(model),
            X_train,
            y_train,
            cv=5,
            method='predict_proba',
            n_jobs=-1
        )[:, 1]
    
        # ROC
        fpr, tpr, thresholds = roc_curve(y_train, oof_proba)
    
        # Youden index
        youden_index = tpr - fpr
    
        best_idx = np.argmax(youden_index)
    
        best_threshold = thresholds[best_idx]
    
        print(f"Best Threshold = {best_threshold:.4f}")
        print(f"Sensitivity = {tpr[best_idx]:.4f}")
        print(f"Specificity = {(1-fpr[best_idx]):.4f}")
    
        return float(best_threshold)
    
    def run_all(self):       
        X_train, X_test, y_train, y_test = self.load_data()
        
        print(f"\n>>> 原始資料 筆數...")
        print("y_train", Counter(y_train))
        print("y_test", Counter(y_test))
        print(f"Train samples: {len(y_train)}")
        print(f"Test samples : {len(y_test)}")
        
        # ====== 儲存 Train / Test 原始資料 ======
        train_df = pd.concat([X_train, y_train], axis=1)
        test_df = pd.concat([X_test, y_test], axis=1)
        
        train_path = os.path.join(self.dirs['data'], f'train_dataset_{Y_LABEL}_{self.feature_name}.csv')
        test_path = os.path.join(self.dirs['data'], f'test_dataset_{Y_LABEL}_{self.feature_name}.csv')

        train_df.to_csv(train_path, index=False)
        test_df.to_csv(test_path, index=False)

        print("✔ 已儲存 Train/Test 資料集")
        
        base_configs = {
            "lightGBM": (
                lgb.LGBMClassifier(),
                "LGBM_grid",
            ),
            "XGBoost": (
                XGBClassifier(),
                "XGB_grid",
            ),
        }

        best_models = {}
        best_params_summary = []
        self.final_thresholds = {}

        for name, (model, grid_key) in base_configs.items():
            search_space = self.cfg["params"][grid_key]
            param_grid = (
                build_lgbm_param_grid(search_space)
                if name == "lightGBM"
                else search_space
            )

            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1,
            )
            grid_search.fit(X_train, y_train)
            best_models[name] = grid_search.best_estimator_

            if self.cfg["threshold"] is None:
                best_threshold = self.find_best_threshold(
                    grid_search.best_estimator_, X_train, y_train
                )
            else:
                best_threshold = self.cfg["threshold"]
            self.final_thresholds[name] = best_threshold

            best_params_summary.append({
                "Model": name,
                **grid_search.best_params_,
            })
            print("=" * 100)
            print(f"{name} best parameters: {grid_search.best_params_}")
            print(f"{name} final threshold: {best_threshold:.4f}")

        # 4. 評估與輸出
        summary_youden = []
        summary_05 = []   
        combined_results = {'Train': {}, 'Test': {}}
        prob_df = pd.DataFrame({"Actual": y_test.reset_index(drop=True)})

        for name, model in best_models.items():
            print(f" >>> 正在處理: {name}")
            if self.cfg['switches']['pkl']:
                pkl_path = os.path.join(self.dirs['pkl'], f'{name}_best_{Y_LABEL}_{self.feature_name}.pkl')
                joblib.dump(model, pkl_path)
                print(f"   ----->已儲存模型 PKL：{pkl_path}")

            for label, X_d, y_t in [('Train', X_train, y_train), ('Test', X_test, y_test)]:
                y_proba_raw = model.predict_proba(X_d)
                y_proba = y_proba_raw[:, 1] if y_proba_raw.ndim > 1 else y_proba_raw

                if label == 'Test': combined_results[label][name] = (y_t, y_proba)
                if self.cfg['switches']['metrics']: 
                    summary_youden.append(self.get_metrics(y_t,y_proba,label,name,threshold=self.final_thresholds[name]))
                    summary_05.append(self.get_metrics( y_t,y_proba,label,name,threshold=0.5))
                if self.cfg['switches']['plots'] and label == 'Test':
                    self._save_visuals(y_t,y_proba,name,label,threshold=self.final_thresholds[name],suffix="Youden")
                    self._save_visuals(y_t,y_proba,name,label,threshold=0.5,suffix="0.5")
                if self.cfg['switches']['proba'] and label == 'Test':
                    prob_df[name] = y_proba
                          
                if label == 'Test' and self.cfg['switches'].get('shap', False):
                    self._plot_shap(model, X_d, name)
                

        # 5. 繪製綜合圖
        if self.cfg['switches']['plots']:
            self._plot_combined_curves(combined_results['Test'], 'Test')
        if self.cfg['switches']['proba']:
            prob_path = os.path.join(
                self.dirs['prob'],f'{Y_LABEL}_All_Model_Probabilities_{self.feature_name}.csv')
            prob_df.to_csv(prob_path, index=False)
            print(f"✔ 已儲存所有模型預測機率：{prob_path}")
        report_youden = pd.DataFrame(summary_youden)
        report_05 = pd.DataFrame(summary_05)
        display(report_youden)
        display(report_05)
        # Save performance and selected-parameter tables.
        pd.DataFrame(best_params_summary).to_csv(os.path.join(self.dirs['perf'], f'{Y_LABEL}_best_params_{self.feature_name}.csv'), index=False)
        
        if self.cfg['save']['metrics']:
            report_youden.to_csv(os.path.join(self.dirs['perf'],f'{Y_LABEL}_total_report_Youden_{self.feature_name}.csv'),index=False)
            report_05.to_csv(os.path.join(self.dirs['perf'],f'{Y_LABEL}_total_report_Threshold_0.5_{self.feature_name}.csv'),index=False)
        print("\n" + "="*100)
        print(f"結果儲存於目錄: {self.output_dir}")
        print("="*100)
        
        return report_youden, report_05, best_models, X_test, y_test


## 4. Run the modeling pipeline


In [ ]:
# --- Run ---
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Authorized input data were not found at {DATA_PATH}. "
        "See data/README.md for the required local file location."
    )

if __name__ == "__main__":
    manager = ModelManager(
        GLOBAL_CONFIG,
        str(DATA_PATH),
        str(OUTPUT_ROOT)
    )
    report_youden, report_05, best_models, X_test, y_test = manager.run_all()



## 5. Calibration analysis and calibration plots


In [ ]:
# ================================================================
# Reviewer-requested calibration analysis
# Calibration fitting: training set only (5-fold CV)
# Final assessment: held-out test set only
# Calibrated models are NOT saved/deployed
# ================================================================
import statsmodels.api as sm
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold

# Reload the exact training set saved by run_all().
train_file = os.path.join(
    manager.dirs['data'],
    f'train_dataset_{Y_LABEL}_{manager.feature_name}.csv'
)
train_calibration_df = pd.read_csv(train_file)
X_train_calibration = train_calibration_df[X_FEATURES]
y_train_calibration = train_calibration_df[Y_LABEL]

training_prevalence = float(y_train_calibration.mean())
test_prevalence = float(np.asarray(y_test).mean())
calibration_cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=GLOBAL_CONFIG['random_state']
)

calibration_rows = []
calibration_plot_data = {}
calibration_probability_df = pd.DataFrame({
    'Actual': pd.Series(y_test).reset_index(drop=True)
})

for model_name in ['lightGBM', 'XGBoost']:
    if model_name not in best_models:
        continue

    print(f'\nCalibration analysis: {model_name}')
    raw_model = best_models[model_name]
    raw_test_proba = raw_model.predict_proba(X_test)[:, 1]

    # Platt/sigmoid calibration fitted exclusively from training data.
    try:
        calibrated_model = CalibratedClassifierCV(
            estimator=clone(raw_model),
            method='sigmoid',
            cv=calibration_cv,
            n_jobs=-1
        )
    except TypeError:
        calibrated_model = CalibratedClassifierCV(
            base_estimator=clone(raw_model),
            method='sigmoid',
            cv=calibration_cv
        )

    calibrated_model.fit(X_train_calibration, y_train_calibration)
    calibrated_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

    # Brier scores and Brier skill score
    brier_uncalibrated = brier_score_loss(y_test, raw_test_proba)
    brier_calibrated = brier_score_loss(y_test, calibrated_test_proba)
    null_proba = np.full(len(y_test), training_prevalence, dtype=float)
    brier_null = brier_score_loss(y_test, null_proba)
    brier_skill_score = 1.0 - (brier_calibrated / brier_null)

    # Observed events, expected events, and O:E ratio
    observed_events = int(np.asarray(y_test).sum())
    expected_events = float(calibrated_test_proba.sum())
    oe_ratio = observed_events / expected_events if expected_events > 0 else np.nan

    # Calibration intercept and slope on held-out test set
    eps = 1e-6
    p_clipped = np.clip(calibrated_test_proba, eps, 1.0 - eps)
    predicted_logit = np.log(p_clipped / (1.0 - p_clipped))
    design_matrix = sm.add_constant(predicted_logit)
    try:
        recalibration_model = sm.Logit(
            np.asarray(y_test, dtype=int), design_matrix
        ).fit(disp=False)
        calibration_intercept = float(recalibration_model.params[0])
        calibration_slope = float(recalibration_model.params[1])
    except Exception as exc:
        print(f'Calibration intercept/slope could not be estimated: {exc}')
        calibration_intercept = np.nan
        calibration_slope = np.nan

    metric_row = report_youden.loc[
        (report_youden['Model'] == model_name) &
        (report_youden['Dataset'] == 'Test')
    ]
    selected_threshold = manager.final_thresholds[model_name]
    test_f1 = float(metric_row['F1'].iloc[0]) if not metric_row.empty else np.nan

    calibration_rows.append({
        'Outcome': Y_LABEL,
        'Model': model_name,
        'Calibration_method': 'Sigmoid calibration (Platt scaling)',
        'Calibration_fitting_data': '5-fold CV within training set',
        'Evaluation_data': 'Held-out test set',
        'Training_prevalence': training_prevalence,
        'Test_prevalence': test_prevalence,
        'Brier_uncalibrated': brier_uncalibrated,
        'Brier_calibrated': brier_calibrated,
        'Brier_null': brier_null,
        'Brier_skill_score': brier_skill_score,
        'Calibration_intercept': calibration_intercept,
        'Calibration_slope': calibration_slope,
        'Observed_events': observed_events,
        'Expected_events': expected_events,
        'O_E_ratio': oe_ratio,
        'Youden_threshold_from_training_CV': selected_threshold,
        'Test_F1_at_Youden_threshold': test_f1
    })

    calibration_probability_df[f'{model_name}_uncalibrated'] = raw_test_proba
    calibration_probability_df[f'{model_name}_calibrated'] = calibrated_test_proba
    calibration_plot_data[model_name] = calibrated_test_proba

    print(f'  Uncalibrated Brier: {brier_uncalibrated:.4f}')
    print(f'  Calibrated Brier:   {brier_calibrated:.4f}')
    print(f'  Null Brier:         {brier_null:.4f}')
    print(f'  Brier skill score:  {brier_skill_score:.4f}')
    print(f'  Calibration intercept: {calibration_intercept:.4f}')
    print(f'  Calibration slope:     {calibration_slope:.4f}')
    print(f'  Observed events:       {observed_events}')
    print(f'  Expected events:       {expected_events:.2f}')
    print(f'  O:E ratio:             {oe_ratio:.4f}')
    print(f'  Training-CV Youden threshold: {selected_threshold:.4f}')
    print(f'  Test F1: {test_f1:.3f}')

    # Do not save the calibrated model.
    del calibrated_model

# Save calibration statistics
calibration_report = pd.DataFrame(calibration_rows).round(4)
display(calibration_report)
calibration_report_path = os.path.join(
    manager.dirs['perf'],
    f'{Y_LABEL}_Calibration_Report_{manager.feature_name}.csv'
)
calibration_report.to_csv(calibration_report_path, index=False, encoding='utf-8-sig')

calibration_probability_path = os.path.join(
    manager.dirs['prob'],
    f'{Y_LABEL}_Calibration_Probabilities_{manager.feature_name}.csv'
)
calibration_probability_df.to_csv(
    calibration_probability_path, index=False, encoding='utf-8-sig'
)

# ================================================================
# Formal calibration plot: calibrated probabilities only
# Held-out test set; 5 quantile bins
# ================================================================
calibration_plot_dir = os.path.join(OUTPUT_ROOT, 'plot',manager.feature_name)
os.makedirs(calibration_plot_dir, exist_ok=True)

plt.figure(figsize=(10, 6))
colors = {'lightGBM': '#1f77b4', 'XGBoost': '#ff7f0e'}
for model_name, calibrated_test_proba in calibration_plot_data.items():
    prob_true, prob_pred = calibration_curve(
        y_test,
        calibrated_test_proba,
        n_bins=10,
        strategy='uniform'
    )
    plt.plot(
        prob_pred, prob_true,
        marker='o', linewidth=2, markersize=7,
        color=colors.get(model_name), label=model_name
    )

plt.plot(
    [0, 1], [0, 1], '--', color='gray', linewidth=1.5,
    label='Perfect calibration'
)
plt.xlabel('Mean calibrated predicted probability')
plt.ylabel('Observed event proportion')
plt.title(f'Post-calibration plot ({Y_LABEL})')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(loc='best')
plt.grid(alpha=0.3)

calibration_plot_path = os.path.abspath(os.path.join(
    calibration_plot_dir,
    f'{Y_LABEL}_PostCalibration_Test_{manager.feature_name}.png'
))
plt.savefig(calibration_plot_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print('\nCalibration report:', os.path.abspath(calibration_report_path))
print('Report exists:', os.path.isfile(calibration_report_path))
print('Calibration probabilities:', os.path.abspath(calibration_probability_path))
print('Probability file exists:', os.path.isfile(calibration_probability_path))
print('Post-calibration plot:', calibration_plot_path)
print('Plot exists:', os.path.isfile(calibration_plot_path))
print('Note: calibrated models were not saved and do not replace operational models.')
